# Streaming Input on Colab with Live Dashboard

This notebook runs the repo's `streaming_input` runtime on a Google Colab GPU, using an existing benchmark run plus a folder of images as the live stream source.

Important constraint: the current runtime does not restore serialized weights from disk. It rebuilds the selected model from `benchmark_summary.json` and, for most backends, performs a historical fit on the original training split again. The benefit of this notebook is that the fit happens on Colab GPU instead of your local CPU machine.

The live source is still folder-based, not a webcam. The dashboard is exposed inside Colab via a forwarded local port.


In [ ]:
# 1. Mount Drive and define the paths for this session.
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

BENCH_RUN_DIR = '/content/drive/MyDrive/thesis_runs/jobB_deceuninck_20260425_101433'
DATASET_DIR   = '/content/drive/MyDrive/Deceuninck_dataset'
INPUT_DIR     = DATASET_DIR
RESULTS_DIR   = '/content/drive/MyDrive/thesis_runs/streaming'

# Pick one model that exists inside BENCH_RUN_DIR/benchmark_summary.json.
MODEL         = 'anomalib_patchcore'

# Runtime controls.
FIT_POLICY    = 'historical_fit'   # auto | historical_fit | skip_fit
MAX_FRAMES    = '300'
TARGET_FPS    = '10'
PORT          = '8765'
LOOP          = '0'                # '1' for endless loop over INPUT_DIR

os.environ['BENCH_RUN_DIR'] = BENCH_RUN_DIR
os.environ['DATASET_DIR']   = DATASET_DIR
os.environ['INPUT_DIR']     = INPUT_DIR
os.environ['RESULTS_DIR']   = RESULTS_DIR
os.environ['MODEL']         = MODEL
os.environ['FIT_POLICY']    = FIT_POLICY
os.environ['MAX_FRAMES']    = MAX_FRAMES
os.environ['TARGET_FPS']    = TARGET_FPS
os.environ['PORT']          = PORT
os.environ['LOOP']          = LOOP

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
print('BENCH_RUN_DIR =', BENCH_RUN_DIR)
print('DATASET_DIR   =', DATASET_DIR)
print('INPUT_DIR     =', INPUT_DIR)
print('RESULTS_DIR   =', RESULTS_DIR)
print('MODEL         =', MODEL)
print('FIT_POLICY    =', FIT_POLICY)
print('PORT          =', PORT)


In [ ]:
# 2. Validate the benchmark run and list the models available in its summary.
import json
from pathlib import Path

bench_run_dir = Path(os.environ['BENCH_RUN_DIR'])
dataset_dir = Path(os.environ['DATASET_DIR'])
input_dir = Path(os.environ['INPUT_DIR'])
summary_path = bench_run_dir / 'benchmark_summary.json'

assert bench_run_dir.is_dir(), f'BENCH_RUN_DIR not found: {bench_run_dir}'
assert summary_path.is_file(), f'benchmark_summary.json not found: {summary_path}'
assert dataset_dir.is_dir(), f'DATASET_DIR not found: {dataset_dir}'
assert input_dir.is_dir(), f'INPUT_DIR not found: {input_dir}'

summary = json.loads(summary_path.read_text(encoding='utf-8'))
models = [row['model'] for row in summary.get('models', []) if isinstance(row, dict) and 'model' in row]
print('Models in benchmark_summary.json:')
for name in models:
    print(' -', name)

assert os.environ['MODEL'] in models, f"Selected MODEL={os.environ['MODEL']} is not present in benchmark_summary.json"
print('\nHistorical dataset path embedded in summary:')
print(' - path      =', summary.get('dataset', {}).get('path'))
print(' - extract   =', summary.get('dataset', {}).get('extract_dir'))
print('\nThese paths will be overridden at runtime with DATASET_DIR on Colab.')


In [ ]:
# 3. Clone or update the repo on Colab.
REPO_URL = 'https://github.com/LorenzSF/Real-time-visual-defect-detection.git'
REPO_DIR = '/content/Real-time-visual-defect-detection'
BRANCH = 'main'

import os
import subprocess

if os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', '-C', REPO_DIR, 'fetch', '--all'])
    subprocess.check_call(['git', '-C', REPO_DIR, 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])

os.environ['REPO_DIR'] = REPO_DIR
print('Repo ready at', REPO_DIR)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--oneline']).decode().strip())


In [ ]:
# 4. Install system and Python dependencies.
!apt-get -qq update
!apt-get -qq install -y rsync >/dev/null
!pip install -q "anomalib>=2.2,<3" lightning "transformers>=4.46,<5" scikit-learn timm kornia einops FrEIA
!pip install -q -e /content/Real-time-visual-defect-detection --no-deps || echo '[note] editable install skipped (not required)'

import torch, anomalib, lightning, transformers
print('torch       ', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')
print('anomalib    ', anomalib.__version__)
print('lightning   ', lightning.__version__)
print('transformers', transformers.__version__)


In [ ]:
# 5. Launch the streaming runtime in the background.
# The helper script stages the benchmark run and datasets to /content/work,
# then starts runtime_main.py with dataset-path overrides so Colab paths win.

import os
import subprocess
from pathlib import Path

repo_dir = Path(os.environ['REPO_DIR'])
script_path = repo_dir / 'scripts' / 'run_streaming_colab.sh'
launcher_log = Path('/content/work/streaming_launcher.log')
pid_path = Path('/content/work/streaming_colab.pid')
Path('/content/work').mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
with launcher_log.open('w', encoding='utf-8') as handle:
    proc = subprocess.Popen(
        ['bash', str(script_path)],
        cwd=str(repo_dir),
        env=env,
        stdout=handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

pid_path.write_text(str(proc.pid), encoding='utf-8')
print('Streaming launcher PID:', proc.pid)
print('Launcher log        :', launcher_log)
print('Runtime log         :', Path(os.environ['RESULTS_DIR']) / f"streaming_{os.environ['MODEL']}.log")
print('Dashboard URL       :', f"http://127.0.0.1:{os.environ['PORT']}/")


In [ ]:
# 6. Inspect the latest launcher/runtime logs.
import os
import signal
from pathlib import Path

def _pid_is_alive(pid: int) -> bool:
    try:
        os.kill(pid, 0)
    except OSError:
        return False
    return True

pid_path = Path('/content/work/streaming_colab.pid')
launcher_log = Path('/content/work/streaming_launcher.log')
runtime_log = Path(os.environ['RESULTS_DIR']) / f"streaming_{os.environ['MODEL']}.log"

if pid_path.exists():
    pid = int(pid_path.read_text(encoding='utf-8').strip())
    print('PID alive:', _pid_is_alive(pid), '(pid=', pid, ')')
else:
    print('No PID file found yet.')

print('\n=== launcher log ===')
if launcher_log.exists():
    print(launcher_log.read_text(encoding='utf-8')[-12000:])
else:
    print('launcher log not created yet')

print('\n=== runtime log ===')
if runtime_log.exists():
    print(runtime_log.read_text(encoding='utf-8')[-12000:])
else:
    print('runtime log not created yet')


In [ ]:
# 7. Open the live dashboard inside Colab.
# If the page is blank, wait a bit longer and rerun the previous log cell.
from google.colab import output
output.serve_kernel_port_as_iframe(int(os.environ['PORT']), height=1000)


In [ ]:
# 8. Stop the background streaming job.
# Use this when you want to end the session before MAX_FRAMES is reached.
import os
import signal
import time
from pathlib import Path

pid_path = Path('/content/work/streaming_colab.pid')
if not pid_path.exists():
    print('No PID file found.')
else:
    pid = int(pid_path.read_text(encoding='utf-8').strip())
    try:
        os.killpg(pid, signal.SIGTERM)
        print('SIGTERM sent to process group', pid)
        time.sleep(2)
    except ProcessLookupError:
        print('Process already stopped.')


In [ ]:
# 9. Manually sync the latest local streaming session to Drive.
# The helper script does this automatically on normal completion. This cell
# is useful if you stopped the process early and still want the artifacts.
import os
import subprocess
from pathlib import Path

repo_dir = Path(os.environ['REPO_DIR'])
results_dir = Path(os.environ['RESULTS_DIR'])
local_root = repo_dir / 'data' / 'streaming_output'
sessions = sorted(local_root.glob('streaming_output_*'), key=lambda p: p.stat().st_mtime, reverse=True)
assert sessions, f'No local streaming_output_* dirs found under {local_root}'

src = sessions[0]
dest = results_dir / src.name
dest.mkdir(parents=True, exist_ok=True)
subprocess.check_call(['rsync', '-a', f'{src}/', f'{dest}/'])
print('Synced:', src)
print('To    :', dest)


## Notes

- `INPUT_DIR` can be the same folder as `DATASET_DIR`, or a separate folder with the images you want to stream live.
- For your `jobB_deceuninck_20260425_101433` run, valid models are the ones listed in cell 2, typically `anomalib_patchcore`, `anomalib_padim`, and `subspacead`.
- `FIT_POLICY='skip_fit'` will only work if the backend is already warm-started. With the current codebase that is usually not the case, so `historical_fit` is the safe option.
- If you want an endless dashboard demo, set `LOOP='1'` and optionally reduce `TARGET_FPS` to avoid wasting GPU cycles.
- The notebook exposes the dashboard, but the source is still folder-based. There is no webcam ingestion path in this repo.
